In [1]:
import surprise
import pandas as pd
import numpy as np
from surprise.model_selection import GridSearchCV
from surprise.model_selection.split import KFold

In [2]:
ratings=pd.read_csv("D:\\AshleshaRuchika\\PGCP-AI\\Machine Learning\\Cases_Rec_Sys\\filmtrust\\ratings.txt",sep=" ",names=['uid','iid','rating'])
ratings.head()

,uid,iid,rating
0,1,1,2.0
1,1,2,4.0
2,1,3,3.5
3,1,4,3.0
4,1,5,4.0


In [3]:
lowest_rating=ratings["rating"].min()
highest_rating=ratings["rating"].max()
lowest_rating,highest_rating

(0.5, 4.0)

In [4]:
reader=surprise.Reader(rating_scale=(lowest_rating,highest_rating))
data=surprise.Dataset.load_from_df(ratings,reader)   

In [5]:
similarity_options={'name':'cosine','user_based':True}
algo=surprise.KNNBasic(sim_options=similarity_options)
output=algo.fit(data.build_full_trainset())

Computing the cosine similarity matrix...
Done computing similarity matrix.


## tuning for best k

In [6]:
param_grid={"k":[20,30,50,70,90],"user_based":[True]}
kFold=KFold(n_splits=5,random_state=26,shuffle=True)
gs=GridSearchCV(surprise.KNNBasic,param_grid,measures=["rmse","mae"],cv=kFold)
gs.fit(data)

Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computi

In [7]:
print(gs.best_score["rmse"])
print(gs.best_params["mae"])


0.8642793798940183
{'k': 50, 'user_based': True}


In [8]:
iid=ratings['iid'].unique()
print(iid)

[   1    2    3 ... 2069 2070 2071]


In [9]:
user=50
u_iid=ratings[ratings['uid']==user]['iid'].unique()
print("List of item rated by user: ",u_iid)
print("No of items rated by user{0}: {1}".format(user,len(u_iid)))
iids_to_predict=np.setdiff1d(iid,u_iid)
print("Items not rated by user or those items for which the expected ratings are to be predicted ",iids_to_predict)

List of item rated by user:  [  8 211   3   2 219 234  12 254 250 207  11 253 236  84  10   7 233  13
   1   5   6 252 241 216 257 206   4 217   9 215 213  17 255 220 121 245
 239 251 235]
No of items rated by user50: 39
Items not rated by user or those items for which the expected ratings are to be predicted  [  14   15   16 ... 2069 2070 2071]


In [10]:
testSet=[[user,iid,0.] for iid in iids_to_predict]
predictions=algo.test(testSet)
exp_ratings=[(predictions[i].iid,predictions[i].est) for i in range(0,len(predictions))]
exp_ratings=pd.DataFrame(exp_ratings,columns=['iid','est_ratings'])
